In [ ]:
import polars as pl
from src.features import ICARELIST, build_features, ARCHIVELIST
from src.train import train_w_folds, TEST_END, TRAIN_END, report
from datetime import datetime, UTC

df = pl.scan_parquet("../data/parquets/sold_listings_20260901.parquet").lazy()
def train_eval(df: pl.LazyFrame):
    df = build_features(df)
    df = df.with_columns(
        pl.col('primary_designer').is_in(ARCHIVELIST).alias('is_archive'),
        pl.col('primary_designer').is_in(ICARELIST).alias('brands_icare')
    )
    start = datetime(2026, 2, 1, tzinfo=UTC)
    #start = start - relativedelta(months=12)
    end = datetime(2026, 3, 1, tzinfo=UTC)
    return train_w_folds(df, start, end)
folds = train_eval(df)

In [ ]:
print(report(folds, ['within_50_250', 'same_titles']))
print(report(folds, ['within_50_250', 'unseen_titles']))
print(report(folds, ['within_50_250', 'unique_titles']))

In [ ]:
from src.features import FEATURES, build_features
import polars as pl

df = pl.read_parquet("../data/parquets/sold_listings_20260901.parquet").lazy()
df = build_features(df)
f_train = df.select(FEATURES).collect()
print(f_train.schema)
f_train_array = f_train.with_columns(pl.col(pl.Categorical).to_physical()).to_numpy()

Predict one Test

In [1]:
from datetime import datetime, timezone, timedelta
import numpy as np
import sys
sys.path.append("../src")
from predict_one import predict_one
import json
import joblib
from pathlib import Path
import lightgbm as lgb

PST = timezone(timedelta(hours=-8))

test_row = {
    "title": "Balenciaga Steroid Derby Boots",
    "department": "menswear",
    "category": "footwear",
    "category_path": "footwear.boots",
    "color": "black",
    "condition": "is_new",
    "size": "11",
    "created_at": datetime(2026, 2, 15, 8, 53, 16, 761000, tzinfo=PST),
    "cover_photo_url": "https://media-assets.grailed.com/prd/listing/temp/0093b0627d51455bb2b2661e57433c1d",
    "location": "United States",
    "photo_count": 4,
    "measurement_count": 0,
    "designer_names": ["Balenciaga"],
}

fold = np.load("../data/folds/embs_tfidf/2026-02.npz")
out = Path(r"D:\GrailedML\TestModel")
booster = lgb.Booster(model_file=str(out / "model.txt"))
pca = joblib.load(out / "pca.joblib")
vectorizer = joblib.load(out / "vectorizer.joblib")
with open(out / "cat_mapping.json") as f:
    mapping = json.load(f)
predicted_result = predict_one(test_row, pca=pca, vectorizer=vectorizer, booster=booster, mapping=mapping)
i = np.where(fold['id'] == 489575)[0]
if i.size:
    print(np.exp(fold["y_pred"][i[0]]))
print(predicted_result)

C:\Users\mononoaware\OneDrive\Documents\GitHub\GrailedDataAnalysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


393.79682460006984
394.35542521150165
